# **NYU CSCI-GY 3303 ECE-GA 9483 Spring 2026: Lab 1 Pruning**

Hey everyone, this lab focuses on neural network pruning. Please review Lab 0 first to familiarize yourself with the pruning implementation in Torch. The lab contains five parts, with a total of 100 points:

Module 1: Model Evaluation:

* Explore weight distribution across different layers from the perspective of designing pruning strategies. (10 pts)


Module 2: Magnitude Pruning:

* Implement simple magnitude pruning, perform sensitivity analysis, and design a magnitude pruning strategy. (30 pts)

Module 3: Regularization-Based Pruning:

* Implement a regulairzation-based retrain pipeline and analyze how regularization benefits magnitude pruning. (20 pts)

Module 4: Structured Pruning:

* Implement structured pruning, specifically filterwise pruning.(20 pts)

Module 5: Iterative Pruning:

* Implement pruning mask, apply the mask to gradient, and retrain the pruned model. (20 pts)

If you're using Colab, go to the settings menu (Runtime -> Change runtime type) and select GPU as the hardware accelerator. The best way to run your code is using Colab and it's free. (We will talk about GPU requirement in the next section.)

There are two types of questions in this assignment: coding and short-answer questyions.

**For coding, you only need to complete the section labeled as:**   


```
    ##############################################################################
    #TODO: xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
    ##############################################################################

    ##############################################################################
    #END OF YOUR CODE
    ##############################################################################
```

***Please rename this file as "netid_lab_1.ipynb". Ensure it is submitted to BrightSpace by Mar 1, 2026 at 11:59 PM.***


# GPU Requirement:

This assignment requires **CUDA support**. Please make sure to run it on a **local machine with a GPU** or on **Google Colab** with GPU enabled. We recommend using **Google Colab**, as it provides free access to GPUs.

If you are working on Colab, click the 'Connect' button in the top-right corner. Once connected, go to Runtime->Change runtime type and select GPU as the hardware accelerator.

 Once you are connected to a GPU (either on local machine or Colab), you can run the following command to check the GPU on your machine.

In [ ]:
!nvidia-smi

# Initialization

In the following sections, we will import necessary packages, define model structure, and define some helper functions. Please make sure:

**1.   The codes are running on GPU.**

**2.   DO NOT change any codes in this section.**


In [ ]:
! pip install thop

In [ ]:
import torch
import time
import math
import copy
import random
import logging
import requests
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from tqdm import tqdm
from torch.utils.data import DataLoader
from thop import profile
from thop import clever_format
from torchsummary import summary
from typing import Dict, Callable

assert torch.cuda.is_available()
logging.getLogger('thop').setLevel(logging.WARNING)

In [ ]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

ResNet Model (ResNet18, 34, 50, 101, 152)

## Model Define

In [ ]:
class BasicBlock(nn.Module):
    """
    BasicBlock is the fundamental building block of ResNet.
    It consists of two 3x3 convolutional layers.
    If the input dimensions do not match the output dimensions,
    or the stride is not 1, a shortcut (identity mapping) is used to adjust dimensions.
    """

    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        # First convolution layer with batch normalization
        self.conv1 = nn.Conv2d(
            in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(planes)

        # Second convolution layer with batch normalization
        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)

        # Shortcut connection (identity mapping)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_planes,
                    self.expansion * planes,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(self.expansion * planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        shortcut_out = self.shortcut(x)
        out += shortcut_out
        out = F.relu(out)
        return out


class Bottleneck(nn.Module):
    """
    Bottleneck is the advanced building block used in deeper versions of ResNet (e.g., ResNet-50, ResNet-101).
    It consists of three layers: 1x1 conv, 3x3 conv, and another 1x1 conv.
    The purpose of the bottleneck is to reduce computation by compressing the dimensions before the 3x3 convolution.
    """

    expansion = 4

    def __init__(self, in_planes, planes, stride=1):
        super(Bottleneck, self).__init__()
        # 1x1 convolution to reduce the number of input channels
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)

        # 3x3 convolution for main processing
        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)

        # 1x1 convolution to expand the number of output channels
        self.conv3 = nn.Conv2d(
            planes, self.expansion * planes, kernel_size=1, bias=False
        )
        self.bn3 = nn.BatchNorm2d(self.expansion * planes)

        # Shortcut connection
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_planes,
                    self.expansion * planes,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(self.expansion * planes),
            )

    def forward(self, x):
        # Forward pass through the bottleneck block (1x1 -> 3x3 -> 1x1)
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        # Add the shortcut connection and pass through ReLU again
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    """
    ResNet class that builds the full ResNet architecture using the BasicBlock or Bottleneck block.
    The number of blocks and layers is determined by the `num_blocks` parameter.
    """

    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        # Create the subsequent layers of the network using the blocks
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        # Final fully connected layer to classify the output
        self.linear = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        """
        Creates a residual layer consisting of `num_blocks` residual blocks.
        The first block may have a stride > 1 to downsample the feature maps.
        """
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))

        # Pass through the 4 layers of the network
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)

        # Flatten the output and pass through the fully connected layer
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def ResNet18():
    return ResNet(BasicBlock, [2, 2, 2, 2])


def ResNet34():
    return ResNet(BasicBlock, [3, 4, 6, 3])


def ResNet50():
    return ResNet(Bottleneck, [3, 4, 6, 3])


def ResNet101():
    return ResNet(Bottleneck, [3, 4, 23, 3])


def ResNet152():
    return ResNet(Bottleneck, [3, 8, 36, 3])


In [ ]:
def train(model: nn.Module, train_dataloader: DataLoader, lr=0.1, epochs=10, device="cuda"):
    """
    Training function for training the model using the given dataloader, optimizer, and loss function.

    Args:
        model: The neural network model to be trained.
        train_dataloader: The DataLoader for the training data.
        lr: The learning rate for the optimizer.
        epochs: Number of epochs to train the model.
        device: Device to run the training on ('cuda' or 'cpu').

    Returns:
        None. Prints training progress for each epoch.
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(
        model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs
    )

    for epoch in range(epochs):
        print(f"\nEpoch: {epoch+1}/{epochs}")
        model.train()
        train_loss = 0
        correct = 0
        total = 0

        with tqdm(total=len(train_dataloader), desc=f"Train Epoch {epoch+1}") as pbar:
            for batch_idx, (inputs, targets) in enumerate(train_dataloader):
                inputs, targets = inputs.to(device), targets.to(device)

                optimizer.zero_grad()
                outputs = model(inputs)

                # Compute the loss
                loss = criterion(outputs, targets)

                # Backward pass (backpropagation)
                loss.backward()

                # Optimizer step (update model parameters)
                optimizer.step()

                # Update the running loss and accuracy
                train_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()

                pbar.set_postfix(
                    {
                        "Loss": f"{train_loss/(batch_idx+1):.3f}",
                        "Acc": f"{100.0 * correct / total:.3f}%",
                    }
                )
                pbar.update(1)
        scheduler.step()

        epoch_loss = train_loss / len(train_dataloader)
        epoch_acc = 100.0 * correct / total
        print(
            f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%"
        )


Download and model initialization function

In [ ]:
def download_and_load_model(url: str, weights_path: str, device="cuda"):
    """
    Downloads the model weights from the given URL and loads them into a ResNet18 model.

    Args:
        url (str): URL to download the model weights from.
        weights_path (str): Path to save the downloaded weights.
        device (str): Device to load the model onto ('cuda' or 'cpu').

    Returns:
        model (torch.nn.Module): The ResNet18 model with the loaded weights.
        recover_model (lambda): A lambda function to recover the model's state.
    """
    print(f"Downloading weights from {url}...")
    r = requests.get(url, allow_redirects=True)
    open(weights_path, 'wb').write(r.content)

    model = ResNet18()

    print(f"Loading weights from {weights_path}...")
    checkpoint = torch.load(weights_path, map_location=torch.device(device))

    if 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'])
    else:
        model.load_state_dict(checkpoint)

    def recover_model():
        print("Recovering model to its original state.")
        if 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'], strict = False)
        else:
            model.load_state_dict(checkpoint, strict=False)

    model = model.to(device)
    return model, recover_model

## Dataset

In [ ]:
class CIFAR:
    """
    Initialize CIFAR dataset (CIFAR-10 in this case).
    This class prepares the train and test data loaders.
    """

    def __init__(self, batch_size=128, num_workers=2):
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.train_loader, self.test_loader = self.prepare_data()

    def prepare_data(self):
        """
        Prepare train and test data loaders with transforms.
        """
        transform_train = transforms.Compose(
            [
                transforms.RandomCrop(32, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(
                    (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)
                ),
            ]
        )

        transform_test = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Normalize(
                    (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)
                ),
            ]
        )

        trainset = torchvision.datasets.CIFAR10(
            root="./data", train=True, download=True, transform=transform_train
        )
        trainloader = torch.utils.data.DataLoader(
            trainset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )

        testset = torchvision.datasets.CIFAR10(
            root="./data", train=False, download=True, transform=transform_test
        )
        testloader = torch.utils.data.DataLoader(
            testset, batch_size=128, shuffle=False, num_workers=self.num_workers
        )

        return trainloader, testloader

## Helper Function

In [ ]:
def test_magnitude_prune(
    test_tensor=torch.tensor([[0.15, -0.22, 0.75, -0.50, 0.60],
                              [-0.05, 0.82, -0.33, 0.19, -0.72],
                              [0.99, 0.03, -0.44, 0.85, -0.13],
                              [0.21, -0.11, 0.45, -0.78, 0.07],
                              [-0.29, 0.53, 0.89, -0.35, 0.46]]),
    test_mask=torch.tensor([[False, False, True, True, True],
                            [False, True, False, False, True],
                            [True, False, False, True, False],
                            [False, False, False, True, False],
                            [False, True, True, False, False]]),
    target_sparsity=0.6):

    def plot_matrix(tensor, ax, title):
        ax.imshow(tensor.cpu().numpy() == 0, vmin=0, vmax=1, cmap='summer')
        ax.set_title(title)
        ax.set_yticklabels([])
        ax.set_xticklabels([])
        for i in range(tensor.shape[1]):
            for j in range(tensor.shape[0]):
                text = ax.text(j, i, f'{tensor[i, j].item():.2f}', ha="center", va="center", color="k")

    # Clone the test tensor to avoid modifying the original
    test_tensor = test_tensor.clone()

    # Plot the original dense tensor
    fig, axes = plt.subplots(1, 2, figsize=(6, 10))
    ax_left, ax_right = axes.ravel()
    plot_matrix(test_tensor, ax_left, 'Dense Tensor')

    # Perform magnitude pruning
    pruned_tensor, mask = magnitude_pruning(test_tensor, target_sparsity)

    # Plot the pruned (sparse) tensor
    plot_matrix(pruned_tensor, ax_right, 'Sparse Tensor')
    fig.tight_layout()
    plt.show()

    # Print the results
    print('* Test magnitude_prune()')
    print(f'    Target sparsity: {target_sparsity:.2f}')
    print(f'    Sparsity of pruning mask: {mask.float().mean():.2f}')
    print(f'    Sparsity of pruned tensor: {(pruned_tensor == 0).float().mean():.2f}')
    if target_sparsity == 0.6:
      if test_mask.equal(mask):
          print('* Test passed.')
      else:
          print('* Test failed.')
    else:
      if (mask.float().mean() == 1 - target_sparsity):
          print('* Test passed.')
      else:
          print('* Test failed.')

In [ ]:
def test_loss_sensitive_magnitude_prune(
    test_tensor=torch.tensor([[0.15, -0.22, 0.75, -0.50, 0.60],
                              [-0.05, 0.82, -0.33, 0.19, -0.72],
                              [0.99, 0.03, -0.44, 0.85, -0.13],
                              [0.21, -0.11, 0.45, -0.78, 0.07],
                              [-0.29, 0.53, 0.89, -0.35, 0.46]]),
    test_grad=torch.tensor([[5.0,  0.01, 0.01,  0.01,  5.0],
                            [0.01, 0.01, 5.0,   5.0,   0.01],
                            [0.01, 5.0,  0.01,  0.01,  5.0],
                            [5.0,  0.01, 5.0,   0.01,  0.01],
                            [0.01, 5.0,  0.01,  5.0,   0.01]]),
    test_mask_06=torch.tensor([[ True, False, False, False,  True],
                              [False, False,  True,  True, False],
                              [False,  True, False, False,  True],
                              [ True, False,  True, False, False],
                              [False,  True, False,  True, False]]),
    test_mask_04=torch.tensor([[True, False, False, False, True],
                               [False, True, True, True, False],
                               [True, True, False, True, True],
                               [True, False, True, True, False],
                               [False, True, True, True, False]]),
    target_sparsity=0.6):

    def plot_matrix(tensor, ax, title):
        ax.imshow(tensor.cpu().numpy() == 0, vmin=0, vmax=1, cmap='summer')
        ax.set_title(title)
        ax.set_yticklabels([])
        ax.set_xticklabels([])
        for i in range(tensor.shape[1]):
            for j in range(tensor.shape[0]):
                text = ax.text(j, i, f'{tensor[i, j].item():.2f}', ha="center", va="center", color="k")

    # Clone the test tensor to avoid modifying the original
    test_tensor = test_tensor.clone()

    # Plot the original dense tensor
    fig, axes = plt.subplots(1, 2, figsize=(6, 10))
    ax_left, ax_right = axes.ravel()
    plot_matrix(test_tensor, ax_left, 'Dense Tensor')

    # Perform magnitude pruning
    pruned_tensor, mask = magnitude_pruning_gradient_based(test_tensor, test_grad, target_sparsity)

    # Plot the pruned (sparse) tensor
    plot_matrix(pruned_tensor, ax_right, 'Sparse Tensor')
    fig.tight_layout()
    plt.show()

    # Print the results
    print('* Test magnitude_prune()')
    print(f'    Target sparsity: {target_sparsity:.2f}')
    print(f'    Sparsity of pruning mask: {mask.float().mean():.2f}')
    print(f'    Sparsity of pruned tensor: {(pruned_tensor == 0).float().mean():.2f}')
    if target_sparsity == 0.6:
      if test_mask_06.equal(mask):
          print('* Test passed.')
      else:
          print('* Test failed.')
    elif target_sparsity == 0.4:
      if test_mask_04.equal(mask):
          print('* Test passed.')
      else:
          print('* Test failed.')
    else:
      if (mask.float().mean() == 1 - target_sparsity):
          print('* Test passed.')
      else:
          print('* Test failed.')

In [ ]:
def plot_layer_sensitivity(layer_accuracies_data, sparsities, baseline=93.79):
    """
    Plot the accuracy loss for different layer groups based on sparsity levels.

    Args:
        layer_accuracies_data (dict): A dictionary with layer names as keys and accuracy lists as values.
        baseline (float): The baseline accuracy of the unpruned model.
        sparsities (array-like): The sparsity levels applied for pruning.
    """
    layer_accuracies_loss = {}

    for layer in layer_accuracies_data:
        layer_accuracies_loss[layer] = [(acc - baseline) for acc in layer_accuracies_data[layer]]

    layer_groups = {
        'layer1': [i for i in layer_accuracies_loss.keys() if 'layer1' in i],
        'layer2': [i for i in layer_accuracies_loss.keys() if 'layer2' in i],
        'layer3': [i for i in layer_accuracies_loss.keys() if 'layer3' in i],
        'layer4': [i for i in layer_accuracies_loss.keys() if 'layer4' in i]
    }

    markers = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', 'h', 'H', 'x', 'd']

    fig, axs = plt.subplots(2, 2, figsize=(12, 8))
    axs = axs.flatten()

    for i, (layer_group, layers) in enumerate(layer_groups.items()):
        ax = axs[i]
        for j, layer in enumerate(layers):
            marker = markers[j % len(markers)]
            ax.plot(sparsities, layer_accuracies_loss[layer], marker=marker, label=layer)

        ax.set_title(f'Layer Group: {layer_group}')
        ax.set_xlabel('Sparsity')
        ax.set_ylabel('Accuracy Loss')
        ax.legend()
        ax.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_reg_model_accuracy_change(layer_accuracies_data_old, layer_accuracies_data_new, sparsities):
    layer_accuracies_diff = {}

    for layer in layer_accuracies_data_old:
        old_accs = layer_accuracies_data_old[layer]
        new_accs = layer_accuracies_data_new[layer]
        layer_accuracies_diff[layer] = [(new - old) for new, old in zip(new_accs, old_accs)]

    layer_groups = {
        'Layer1': sorted([i for i in layer_accuracies_diff.keys() if 'layer1' in i]),
        'Layer2': sorted([i for i in layer_accuracies_diff.keys() if 'layer2' in i]),
        'Layer3': sorted([i for i in layer_accuracies_diff.keys() if 'layer3' in i]),
        'Layer4': sorted([i for i in layer_accuracies_diff.keys() if 'layer4' in i])
    }

    plt.style.use('default')
    fig, axs = plt.subplots(2, 2, figsize=(15, 10))
    axs = axs.flatten()

    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FECA57', '#FF9FF3', '#54A0FF', '#5F27CD']

    for i, (layer_group, layers) in enumerate(layer_groups.items()):
        ax = axs[i]

        if not layers:
            ax.text(0.5, 0.5, f'No {layer_group} layers found',
                   ha='center', va='center', transform=ax.transAxes, fontsize=14)
            ax.set_title(f'{layer_group}', fontsize=16, fontweight='bold', pad=20)
            continue

        n_sparsities = len(sparsities)
        n_layers = len(layers)
        bar_width = 0.15
        x_positions = np.arange(n_sparsities)

        for j, layer in enumerate(layers):
            offset = (j - (n_layers - 1) / 2) * bar_width
            diff_values = layer_accuracies_diff[layer]

            bars = ax.bar(x_positions + offset, diff_values, bar_width,
                         label=layer, alpha=0.8, color=colors[j % len(colors)],
                         edgecolor='white', linewidth=0.5)

        ax.set_title(f'{layer_group} - Accuracy Difference', fontsize=14, fontweight='bold', pad=15)
        ax.set_xlabel('Sparsity Level', fontsize=12)
        ax.set_ylabel('Accuracy Difference (%)', fontsize=12)

        ax.set_xticks(x_positions)
        ax.set_xticklabels([f'{s:.1f}' for s in sparsities], fontsize=10)

        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9,
                 frameon=True, fancybox=True, shadow=True)

        ax.grid(True, alpha=0.3, linestyle='--')
        ax.axhline(y=0, color='black', linestyle='-', linewidth=1.5, alpha=0.8)

        if layers:
            y_min = min([min(layer_accuracies_diff[layer]) for layer in layers])
            y_max = max([max(layer_accuracies_diff[layer]) for layer in layers])
            margin = (y_max - y_min) * 0.1 if y_max != y_min else 0.1
            ax.set_ylim(y_min - margin, y_max + margin)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

## Load Model and Dataset

In [ ]:
url = "https://github.com/shawnyin128/25Fall-CSCI-GA-3033-EfficientAI-Materials/blob/master/model/best_model.pth?raw=true"
weights_path = "./best_model.pth"

model, recover_model = download_and_load_model(url, weights_path, device="cuda")

In [ ]:
dataloader = {}
dataloader['train'], dataloader['test'] = CIFAR().train_loader, CIFAR().test_loader

# Module 1: Model Evaluation (10 pts)

## Question 1.1: Model Metrics (3 pts)

**Context & Motivation:**

Model metrics are essential for providing an objective standard to evaluate success and guide our optimization. In this lab, our analysis focuses on the trade-off between two key dimensions: Performance and Efficiency.

**Methods & Concepts:**

For Performance, since we are performing classification with ResNet, Accuracy is the most direct metric to assess the model's predictive capability. For Efficiency, we use Parameter Count and MACs (Multiply-Accumulate Operations) to directly measure the model's size and computational workload. Furthermore, as we apply pruning, we will use Sparsity to quantify the proportion of redundant weights successfully removed.


**Tasks:**

In this question, you will need to implement those four metrics.

*  **Accuracy**: The percentage of correct predictions on the test set.
*  **Sparsity**: The proportion of weights in the model that are zero, indicating how sparse the model is.
*  **MACs (Multiply-Accumulate Operations)**: A measure of the computational complexity of the model.
*  **Parameters**: The total number of learnable parameters in the model.


In [ ]:
def evaluate(model: nn.Module, test_dataloader: DataLoader, device="cuda") -> float:
    """
    Inference function to evaluate the model on a test dataset using the provided dataloader,
    with tqdm progress bar for visualization.

    Args:
        model: The neural network model to be used for inference.
        test_dataloader: The dataloader for the test data.
        device: Device to use ('cuda' or 'cpu').

    Returns:
        None: Prints the accuracy on the test set.
    """
    model = model.to(device)
    model.eval()

    correct = 0
    total = 0

    # Disable gradient computation during inference
    with torch.no_grad():
        for inputs, targets in tqdm(test_dataloader, desc="Evaluating", leave=False):
            inputs, targets = inputs.to(device), targets.to(device)

            ##############################################################################
            #TODO:
            #1. forward
            #2. get the result from model outputs. (function you may need to use: max())
            ##############################################################################


            ##############################################################################
            #END OF YOUR CODE
            ##############################################################################

            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    accuracy = 100.0 * correct / total
    # print(f"Accuracy on the test set: {accuracy:.2f}%")
    return accuracy

In [ ]:
def calculate_sparsity(model: nn.Module) -> float:
    """
    Calculate the sparsity of a given model (proportion of zero weights).

    Args:
        model (nn.Module): The PyTorch model to calculate sparsity for.

    Returns:
        float: The sparsity of the model (proportion of zero weights).
    """
    total_params = 0
    zero_params = 0

    # Iterate over each parameter tensor in the model
    for param in model.parameters():
        ##############################################################################
        #TODO:
        #1. accumulate total_params by the current number of parameters
        #2. accumulate zero_params by the current number of zero parameters (here you will only care about param strictly equal to 0)
        ##############################################################################



        ##############################################################################
        #END OF YOUR CODE
        ##############################################################################

    sparsity = zero_params / total_params
    return sparsity

In [ ]:
def calculate_MACs_and_Params_by_thop(model: nn.Module, input_size: tuple = (1, 3, 32, 32)) -> float:
    """
    Calculate the MACs (Multiply-Accumulate) of a model.

    Args:
        model (nn.Module): The PyTorch model to calculate MACs for.
        input_size (tuple): The size of the input tensor, e.g., (1, 3, 32, 32) for CIFAR-10.

    """
    # Create a dummy input with the correct input size
    dummy_input = torch.randn(*input_size).cuda()

    # Use thop to profile the model and calculate FLOPs
    ##############################################################################
    #TODO:
    # compute the MACs and params using thop. You can check its repo https://github.com/ultralytics/thop for reference.
    ##############################################################################



    ##############################################################################
    #END OF YOUR CODE
    ##############################################################################

    macs, params = clever_format([macs, params], "%.3f")

    return macs, params

Next, run the following cell to evaluate the original model. You will expect:



*   **Accuracy**: As a pre-trained model, it already has strong classification capability. You shuld expect accuracy around 93.79% on test dataset.
*   **Sparsity**: As a dense model, you should expect sparsity equals to 0%.
*   **MACs**: For our given dummy input, you should expect MACs to be 557.88M.
*   **Params**: This model contains 11.174M parameters.


In [ ]:
accuracy = evaluate(model, dataloader['test'])
sparsity = calculate_sparsity(model)
macs, params = calculate_MACs_and_Params_by_thop(model)
print(f'Accuracy: {accuracy}%, MACs: {macs}, Params: {params}, Sparsity: {sparsity}%')

From the evaluation results, our model achieve a ~93% accuracy in the CIFAR-10 test dataset. However, for a relatively simple task CIFAR-10 classification, this model is arguably larger than necessary. Reducing the model's size without sacrificing much accuracy could improve its efficiency, making it more suitable for deployment on resource-constrained devices like mobile phones or embedded systems.



In the upcoming steps of this lab, we will apply pruning techniques to reduce the size of the model, making it more computationally efficient. By pruning away less important weights, we aim to:


*   **Increse sparsity:** Introduce zero weights, reducing the total number of parameters.
*   **Reduce MACs:**: Lower the computational cost for forward passes, making the model faster.
*   **Maintain performance:**: Keep the accuracy as high as possible, while shrinking the model's size.






## Question 1.2: Weight Distribution of ResNet18 (5 pts)

**Context & Motivation:**

Before we proceed to pruning, we must understand the numerical "landscape" of the original model. In the context of model compression, weight values are not just random numbers; they represent the model's learned knowledge. Investigating the distribution of these weights is a fundamental step, as it allows us to visually and statistically identify redundancy—specifically, the concentration of weights near zero that contribute little to the model's final output.

**Methods & Concepts:**

By analyzing the weight distribution (typically via histograms), we can determine a meaningful pruning threshold. This distribution directly informs our pruning strategy.

**Tasks:**

In this question, you need to complete the function to retrieve weight data for each layer. Run the following cell you will expect a set of histograms.

In [ ]:
def visualize_weight_distribution(model, bins=256, exclude_zero=False):
    weight_data = [] # Collect weights to be visulized
    layer_names = []

    for layer_name, param_tensor in model.named_parameters():
        if param_tensor.dim() > 1:  # Only consider weight tensors, ignore biases and batch norm
            ##############################################################################
            #TODO:
            #1. get the parameter tensor.
            #2. append the parameter data to weight_data.
            #tips: plot only suppor numpy format, so you need to find a way to transfer data from tensor to numpy
            ##############################################################################


            ##############################################################################
            #############################END OF YOUR CODE#################################
            ##############################################################################
            layer_names.append(layer_name)  # Collect layer names

    num_layers = len(weight_data)
    columns = 3
    rows = math.ceil(num_layers / columns)

    fig, axes = plt.subplots(rows, columns, figsize=(10, rows * 2))
    axes = axes.flatten()

    for i, (layer_weights, layer_name) in enumerate(zip(weight_data, layer_names)):
        ax = axes[i]
        if exclude_zero:
            layer_weights = layer_weights[layer_weights != 0]
        ax.hist(layer_weights, bins=bins, density=True, color='blue', alpha=0.7)
        ax.set_title(layer_name)
        ax.set_xlabel('Weight Value')
        ax.set_ylabel('Density')

    for j in range(num_layers, len(axes)):
        fig.delaxes(axes[j])
    fig.tight_layout()
    plt.show()

visualize_weight_distribution(model)

## Question 1.3: Weight Distribution Analysis (2 pts)

Based on the distribution, answer the following question:


**What common characteristics do you observe in the weight distributions across different layers? Can these characteristics make the pruning more efficient or difficult? Explain your reasoning.**

**Answer:**

# Module 2: Magnitude Pruning (30 pts)

Magnitude pruning is a commonly used technique to reduce the size and complexity of DNNs while maintaining good performance. It involves removing weights with smaller magnitudes because these weights contribute less to the output of the model.

By removing these less significant weights, we can reduce the computational and memory requirements of the model without heavily impacting its accuracy.

<p align="center">
  <img src="https://github.com/shawnyin128/25Fall-CSCI-GA-3033-EfficientAI-Materials/raw/master/graph/magnitude_pruning.png" alt="Magnitude Pruning" width="400"/>
</p>



In this module, you will learn how to implement magnitude pruning to reduce the size and complexity of a neural network while maintaining performance. Additionally, you will explore the sensitivity of each layer to pruning, and finally, you'll have the opportunity to experiment with your own pruning strategy.

## Question 2.1: Magnitude Pruning Implementation (10 pts)


**Context & Motivation:**

The pruning of an entire neural network is inherently composed of the pruning of its individual weight matrices. Consequently, a single weight tensor represents the smallest unit of the pruning process. By first implementing the pruning logic for a single weight and creating a reusable function, we can efficiently scale the process to handle the complex, multi-layered architecture of the entire model.

**Methods & Concepts:**

We will adopt the simplified magnitude pruning as our core strategy for each weight parameter. The algorithm can be found in this algorithm block.

<p align="center">
  <img src="https://github.com/shawnyin128/25Fall-CSCI-GA-3033-EfficientAI-Materials/raw/master/graph/algo_magnitude.png" alt="Magnitude Pruning Algorithm" width="400"/>
</p>

**Tasks:**

Your task is to implement the magnitude_pruning function. This function must be designed to process a single weight tensor and apply pruning based on a specified sparsity level.

In [ ]:
def magnitude_pruning(tensor: torch.Tensor, sparsity: float) -> torch.Tensor:
    """
    Apply magnitude pruning to a given tensor.

    Args:
        tensor (torch.Tensor): The input tensor to be pruned.
        sparsity (float): The desired spars
    Returns:
        pruned_tensor (torch.Tensor): The pruned tensor.
        mask (torch.Tensor): The pruning mask (where True means the weight is kept, and False means it is pruned).
    """
    sparsity = min(max(0.0, sparsity), 1.0)
    pruned_tensor = tensor.clone()
    mask = torch.zeros_like(tensor, dtype=torch.bool)
    ##############################################################################
    #TODO:
    #1. flatten the tensor
    #2. calculate the pruning threshold
    #3. create a mask
    #4. create the pruned tensor
    ##############################################################################


    ##############################################################################
    #END OF YOUR CODE
    ##############################################################################
    mask = mask.to(tensor.device)
    pruned_tensor = pruned_tensor.to(tensor.device)

    return pruned_tensor, mask

We validate the correctness of our magnitude pruning function by applying it to a dummy tensor.

In [ ]:
test_magnitude_prune(target_sparsity=0.6)
test_magnitude_prune(target_sparsity=0.4)

## Question 2.2: Model Pruning and Sensitivity Analysis (10 pts)

**Context & Motivation:**

As discussed in Question 2.1, pruning an entire model is equivalent to pruning every individual weight within it. However, our previous weight distribution analysis revealed that each weight matrix possesses a unique numerical distribution. Because of these differences, each layer exhibits a different level of sensitivity to sparsity. To achieve the best overall compression, each layer should ideally operate at its own optimal sparsity level.

**Methods & Concepts:**

To determine these per-layer levels, we will use a grid search approach. By systematically varying the sparsity of each individual layer while keeping others constant, we can observe the tolerance pattern of each layer.

**Tasks:**

1. You need to finish sensitivity_scan_per_layer function to include pruning and evaluation in each sparsity level.
2. You will need to run sensitivity_scan_per_layer using our pre-defined arguments. Then plot the sensitivity graph.

In [ ]:
@torch.no_grad()
def sensitivity_scan_per_layer(model: nn.Module, dataloader: torch.utils.data.DataLoader,
                               start: float, step: float, end: float,
                               device: str = 'cuda', verbose=True):
    """
    Perform a sensitivity scan by applying magnitude pruning to each individual layer of the model
    and evaluate the accuracy at different levels of sparsity for each layer.

    Args:
        model (nn.Module): The neural network model to be pruned and evaluated.
        dataloader (DataLoader): DataLoader for the test dataset.
        start (float): Starting sparsity level.
        step (float): Step size for increasing sparsity.
        end (float): Ending sparsity level.
        device (str): Device to use ('cuda' or 'cpu').
        verbose (bool): Whether to print progress and results.

    Returns:
        dict: Dictionary containing sparsity levels and corresponding accuracies for each layer.
    """
    sparsities = np.arange(start, end, step)

    layer_accuracies = {}

    model.to(device)

    for name, param in model.named_parameters():
        if param.dim() > 1 and 'layer' in name:
            print(f"\nScanning layer: {name}")
            param_clone = param.detach().clone()

            accuracy_per_layer = []
            ##############################################################################
            #TODO:
            #1. for each sparsity in sparsities. prune the model using our implemented function (magnitude_pruning)
            #2. for each sparsity, evaluate the accuracy using evaluate() and add the accuracy to accuracy_per_layer variable
            ##############################################################################


            ##############################################################################
            #END OF YOUR CODE
            ##############################################################################
                if verbose:
                    print(f"Sparsity: {sparsity:.2f} | Accuracy: {accuracy:.2f}%")

                param.data.copy_(param_clone)

            layer_accuracies[name] = accuracy_per_layer
    return sparsities, layer_accuracies

After you have finished the sensitivity function, run the code below to compute the sparsity-accuray data.

In [ ]:
recover_model()
sparsities, layer_accuracies = sensitivity_scan_per_layer(model, dataloader['test'], 0.4, 0.1, 1.0)

You've done a great job running the sensitivity scan and pruning the layers of the ResNet18 model. That was no easy task! But now, the reward awaits. We have provided a visualization function that will allow you to see how the accuracy of the model is affected as we increase the sparsity in different layers.

This plot will clearly show you the impact of pruning on different layers, and how the accuracy degrades as more weights are pruned. So sit back, run the visualization function below, and take a closer look at the results of your hard work!


In [ ]:
plot_layer_sensitivity(layer_accuracies, sparsities)

## Question 2.3: Layer-wise Adaptive Model Pruning (10 pts)

**Context & Motivation:**

As discussed in Question 2.2, we want a layer-wise sparsity level to optmize the magnitude pruning performance.

**Methods & Concepts:**

The primary logic for this task is to create a Per-Layer Sparsity Dictionary. This dictionary acts as a configuration map where each key is a layer identifier (layer name) and each value is its corresponding sparsity level.

**Tasks:**

1. Complete layer_dict, and for each key (representing layer name), assign its sparsity level.
2. Complete the pruning for each layer using the layer_dict. You should achieve:
*   **Sparsity**: at least 0.15.
*   **Accuracy**: at least 93%.

In [ ]:
##############################################################################
#TODO: for each layer, enter the sparsity level you want (a float number between 0.0 to 1.0)
##############################################################################
layer_dict = {
    'layer1.0.conv1.weight': 0.0,
    'layer1.0.conv2.weight': 0.0,
    'layer1.1.conv1.weight': 0.0,
    'layer1.1.conv2.weight': 0.0,
    'layer2.0.conv1.weight': 0.0,
    'layer2.0.conv2.weight': 0.0,
    'layer2.0.shortcut.0.weight': 0.0,
    'layer2.1.conv1.weight': 0.0,
    'layer2.1.conv2.weight': 0.0,
    'layer3.0.conv1.weight': 0.0,
    'layer3.0.conv2.weight': 0.0,
    'layer3.0.shortcut.0.weight': 0.0,
    'layer3.1.conv1.weight': 0.0,
    'layer3.1.conv2.weight': 0.0,
    'layer4.0.conv1.weight': 0.0,
    'layer4.0.conv2.weight': 0.0,
    'layer4.0.shortcut.0.weight': 0.0,
    'layer4.1.conv1.weight': 0.0,
    'layer4.1.conv2.weight': 0.0
}
##############################################################################
#END OF YOUR CODE
##############################################################################

In [ ]:
recover_model()
print(f'Sparsity before pruning: {calculate_sparsity(model)}')
##############################################################################
#TODO:
#1. apply the magnitude_pruning() to each parameter according to its defined sparsity
##############################################################################


##############################################################################
#END OF YOUR CODE
##############################################################################
acc = evaluate(model, dataloader['test'])
print()
print(f'after pruning: {calculate_sparsity(model):4f}, acuracy: {acc}%')


# Module 3: Regularization-Based Pruning (20 pts)

Regularization-based pruning leverages additional penalty terms in the loss function to encourage the network to automatically reduce the importance of certain weights. By applying L1 or L2 regularization, the optimization process naturally drives unimportant weights closer to zero, making them candidates for pruning.

<p align="center">
  <img src="https://github.com/shawnyin128/25Fall-CSCI-GA-3033-EfficientAI-Materials/raw/master/graph/regularization.png" alt="Magnitude Pruning" width="1000"/>
</p>

In this module, you will learn how to implement regularization-based pruning by adding L1 regularization or L2 regularization terms to the training loss, observe how regularization affect the sparsity pattern comparing with pure magnitude quantizaiton.

## Question 3.1: L1 Regularization (5 pts)

**Context & Motivation:**

Unlike other methods, $L_1$ puts constant pressure on smaller weights, driving them to become exactly zero. This creates a more redundant weight distribution.

**Methods & Concepts:**

The core approach is to modify the loss function by adding an $L_1$ penalty term, defined as $L(w) + \lambda ||w||_1$. This term penalizes the absolute magnitude of the weights. During backpropagation, this constant pressure drives less important weights toward zero.

**Tasks:**

In this question, you will implement a custom loss function that incorporates $L_1$ regularization. You are required to:

*   Calculate the standard task loss (e.g., Cross-Entropy).
*   Compute the $L_1$ norm of the target weight matrices.
*   Combine them using a scaling factor ($\lambda$) and use this total loss for the model update.


In [ ]:
def l1_regularized_loss(model: torch.nn.Module, output: torch.Tensor, target: torch.Tensor, reg_lambda: float = 1e-5, device: str="cuda"):
  ##############################################################################
  #TODO:
  #1. Compute the cross entropy loss. You can use torch's cross_entropy() function.
  #2. For each parameter, add its l1 norm to regularization_loss.
  ##############################################################################



  ##############################################################################
  #END OF YOUR CODE
  ##############################################################################
  loss = ce_loss + reg_lambda * regularization_loss
  return loss

## Question 3.2: L2 Regularization (5 pts)

**Context & Motivation:**

While $L_1$ regularization promotes exact sparsity by driving weights to zero, $L_2$ regularization aims to minimize the overall energy of the model by penalizing the square of the weights. By comparing $L_2$ with $L_1$, we can observe how different weight landscapes affect the model's sensitivity to subsequent pruning.

**Methods & Concepts:**

The core approach is to modify the loss function by adding an $L_2$ penalty term, defined as $L(w) + \lambda ||w||_2$. This term penalizes the absolute magnitude of the weights. During backpropagation, this constant pressure drives less important weights toward zero.

**Tasks:**

In this question, you will implement a custom loss function that incorporates $L_2$ regularization. You are required to:

*   Calculate the standard task loss (e.g., Cross-Entropy).
*   Compute the $L_2$ norm of the target weight matrices.
*   Combine them using a scaling factor ($\lambda$) and use this total loss for the model update.


In [ ]:
def l2_regularized_loss(model: torch.nn.Module, output: torch.Tensor, target: torch.Tensor, reg_lambda: float = 1e-5, device: str="cuda"):
  ##############################################################################
  #TODO:
  #1. Compute the cross entropy loss. You can use torch's cross_entropy() function.
  #2. For each parameter, add its l2 norm to regularization_loss.
  ##############################################################################


  ##############################################################################
  #END OF YOUR CODE
  ##############################################################################
  loss = ce_loss + reg_lambda * regularization_loss
  return loss

## Question 3.3: Model Fine-tune (5 pts)

**Context & Motivation:**

Defining a loss function is only the first step; to actually modify the model’s weights, the loss must be integrated into a training loop. By embedding our regularized loss into the standard training flow, we enable the model to learn while simultaneously adhering to the structured constraints ($L_1$ or $L_2$) we have imposed.

**Methods & Concepts:**

A typical PyTorch training loop consists of five essential steps:

1.   **Forward**
2.   **Loss Computation**
3.   **Gradient Zeroing**
4.   **Backward**
5.   **Weight Update**

Our regularization-based loss function will affect the second step.



**Tasks:**

You will implement the five steps in regularized_fine_tune function. The loss computation step should be replaced by our regularization-based loss function.

*To make sure checkable solution, we fix the optmizer, learning rate and epochs for you. Please don't change them.*

In [ ]:
def regularized_fine_tune(model: nn.Module,
                          dataloader: DataLoader,
                          loss_funciton: Callable,
                          device: str = 'cuda'):
    model.to(device)
    model.train()

    finetune_params = []
    for name, param in model.named_parameters():
        if param.dim() > 1 and 'layer' in name:
            param.requires_grad = True
            finetune_params.append(param)
        else:
            param.requires_grad = False
    optimizer = torch.optim.SGD(finetune_params, lr=1e-3)

    epochs = 10

    ##############################################################################
    #TODO:
    #1. Iterate epochs and dataloader
    #3. For eatch batch, define the training process
    ##############################################################################



    ##############################################################################
    #END OF YOUR CODE
    ##############################################################################
    model.eval()

## Question 3.4: Sensitivity Analysis (5 pts)

**Context & Motivation:**

After applying regularization, the model's weight distribution has been reshaped. To understand the practical impact of these changes, we need to re-evaluate the model's sensitivity to pruning. This comparison allows us to see if the regularization actually makes the model more robust to pruning.

**Methods & Concepts:**

We will reuse the sparsity analysis methodology established in Question 2.2. The core logic remains the same: we systematically increase the sparsity level and measure the resulting accuracy. The key difference here is the model—we are now analyzing a model that has undergone a training phase with a modified loss function ($L_1$ or $L_2$).

**Tasks:**

You need to run the following codes first and then answer the questions listed below in the last cell of this section.

We will first run the L1 regualrizaiton fine-tuning. Run the next cell to apply fine-tune loop. This will take around 7 mins if you use T4 GPU.

In [ ]:
recover_model()
regularized_fine_tune(model, dataloader['train'], l1_regularized_loss)

Next, run the following sensitivity analysis first and try to understand how regularization-based method helps pruning.

In [ ]:
sparsities_reg, layer_accuracies_reg = sensitivity_scan_per_layer(model, dataloader['test'], 0.4, 0.1, 1.0)

Now, you have finished the sensitivity analysis for our regularization-based pruned model. Let's compare it with the previous sensitivity result. Run the following codes to generate the accuracy comparison graph.

Here are some remarks for the graph:

**1. Each color represents a specific weight tensor.**

**2. Accuracy Difference is defined as:**

$$Accuracy_{reg}-Accuracy_{mag}$$ where Accuracy_reg is the accuracy (%) after we perform pruning on l1-regularized model, and Accuracy_mag is the accuracy (%) after we perform magnitude pruning on original model. For example, if the accuracy difference is 0.7, it means, given the sparsity level, the l1 regularization magnitude pruning result is 0.7% higher than the original magnitude pruning accuracy.

In [ ]:
plot_reg_model_accuracy_change(layer_accuracies, layer_accuracies_reg, sparsities_reg)

The process above illustrates the analysis for $L_1$ regularization. Next, we will follow the same logic, but with a different loss function, to analyze the impact of $L_2$ regularization on the model. Execute the following three cells to implement this process.

In [ ]:
recover_model()
regularized_fine_tune(model, dataloader['train'], l2_regularized_loss)

In [ ]:
sparsities_reg, layer_accuracies_reg = sensitivity_scan_per_layer(model, dataloader['test'], 0.4, 0.1, 1.0)

In [ ]:
plot_reg_model_accuracy_change(layer_accuracies, layer_accuracies_reg, sparsities_reg)

**Question:** Based on your experimental results, which type of regularization performs better in most cases? How do the effects of L1 and L2 regularization on weight distribution differ? How do these distinct effects impact the final pruning performance?

**Answer:**

# Module 4: Structured Pruning (20 pts)

In the previous part of our lab, we implemented and evaluated the performance of unstructured magnitude pruning. While unstructured pruning typically leads to better performance due to its finer granularity, it presents challenges in practical hardware implementations. This is because unstructured pruning can be difficult to optimize for, making it harder to achieve the theoretical improvements in efficiency.

<p align="center">
  <img src="https://github.com/shawnyin128/25Fall-CSCI-GA-3033-EfficientAI-Materials/raw/master/graph/structural.png" alt="structured Pruning" width="1000"/>

In this module, we will shift our focus to **Structured pruning**, specifically **filter** pruning. Structured pruning offers more direct benefits for hardware acceleration by removing entire filter, thus reducing the computational complexity in a manner that is easier to implement on modern hardware.


## Question 4.1: Filter Pruning (20 pts)

**Context & Motivation:**


Logically, structured pruning is very similar to the magnitude pruning we implemented earlier; we still need a criterion to determine which parameters are important. However, the difference is that this criterion operates at the structured level (filters) rather than at a per-element level. Just as we used weight magnitude to identify redundant individual weights, we can apply a similar philosophy to filters to identify which entire 3D structures contribute the least to the model's feature extraction.

**Methods & Concepts:**

Our filter pruning strategy will be based on $L_1$ norm.

1. **L1 Norm Calculation for Each Filter**:

   For each weight filter \( $c_{out}$ \) in the convolutional layer, the L1 norm is computed as:

   $$
   L1(W_{c_{out}}) = \sum_{i=1}^{K} \sum_{j=1}^{K} \sum_{c_{in}=1}^{C_{in}} |W_{c_{out}}(i,j,c_{in})|
   $$

   Where:
   - $$ W_{c_{out}} \in \mathbb{R}^{K \times K \times C_{in}} \text{ is the weight of output filter }  c_{out} $$
   - $$ K \times K \text{ is the spatial size of the filter, and } C_{in} \text{ is the number of input channels} $$

2. **Pruning Based on L1 Norms**:

   After computing the L1 norm for each filter, the filters with the smaller L1 norms are pruned. Given a pruning ratio  p, the number of retained filters  C_retained is:

   $$
   C_{\text{retained}} = \left( 1 - p \right) \times C
   $$

   Where C  is the total number of weight filters.

3. **Filter Pruning Process**:

   The filters with the lower L1 norms are pruned, leaving only the top C_retained filters with the highest L1 norms.

   This pruning process ensures that only C_retained filters are kept.


**Tasks:**

You need to complete prune_conv_bn_layer function where we prune both weight matrix and batchnorm parameters.

In [ ]:
import torch
import torch.nn as nn
import copy

class ChannelPruner:
    def __init__(self, model: nn.Module, prune_ratio: float):
        """
        Initialize the pruner.

        Args:
            model (nn.Module): The model to be pruned.
            prune_ratio (float): The ratio of filters to prune (0.0 to 1.0).
        """
        self.model = copy.deepcopy(model)
        self.prune_ratio = prune_ratio

    def prune_conv_bn_layer(self, conv_layer: nn.Conv2d, bn_layer: nn.BatchNorm2d):
        """
        Prune a Conv2D layer and its corresponding BatchNorm2d layer based on the L1 norm.

        Args:
            conv_layer (nn.Conv2d): The Conv2D layer to prune.
            bn_layer (nn.BatchNorm2d): The BatchNorm2d layer corresponding to the Conv2D layer.

        Returns:
            mask (torch.Tensor): A boolean mask of the filters kept.
        """
        with torch.no_grad():
            ##############################################################################
            #TODO:
            #1: Calculate the L1 norm for each weight's output channel (filter)
            #2: Sort the filters by L1 norm and create a mask for the remaining filters
            #3: In-place prune the convolution layer weights
            ##############################################################################


            ##############################################################################
            #END OF YOUR CODE
            ##############################################################################

            # Prune the BatchNorm2d layer in-place
            bn_layer.weight.data[prune_indices] = 0
            bn_layer.bias.data[prune_indices] = 0
            bn_layer.running_mean[prune_indices] = 0
            bn_layer.running_var[prune_indices] = 0


        return mask

    def prune_model(self):
        """
        Apply filter pruning to all Conv2D layers in the model and their corresponding BatchNorm2d layers.

        Returns:
            pruned_model (nn.Module): The pruned model.
        """
        previous_mask = None  # Mask from the previous layer
        shortcut_mask = None  # Mask for the shortcut connection
        idx = 1
        for name, layer in self.model.named_modules():
            if isinstance(layer, nn.Conv2d) and 'layer' in name:
                layer_idx = int(name.split('.')[0][-1])
                if layer_idx != idx:
                    idx = layer_idx
                    shortcut_mask = previous_mask
                # Get the corresponding BatchNorm2d layer
                if 'shortcut' not in name:
                    bn_layer_name = name.replace('conv', 'bn')
                else:
                    bn_layer_name = name.replace('shortcut.0', 'shortcut.1')
                    previous_mask = shortcut_mask

                bn_layer = dict(self.model.named_modules()).get(bn_layer_name, None)

                if bn_layer is not None:
                    # If there is a mask from the previous layer, apply it to the input channels
                    if previous_mask is not None:
                        # Ensure the mask is applied to the correct dimension of the weight tensor
                        # layer.weight = nn.Parameter(layer.weight[:, previous_mask, :, :])
                        layer.weight.data[:, ~previous_mask, :, :] = 0  # Set pruned input filters to zero


                    # Prune the current Conv2D and BatchNorm2D layers and get the new mask
                    mask = self.prune_conv_bn_layer(layer, bn_layer)

                    # Update the mask for the next layer (output channels -> next layer's input channels)
                    previous_mask = mask

        return self.model


Next, we will apply the structured pruning and check the pruned model accuracy.

In [ ]:
recover_model()
pruner = ChannelPruner(model, prune_ratio=0.1)
pruned_model = pruner.prune_model()

In [ ]:
acc = evaluate(pruned_model, dataloader['test'])
print(f"\nAfter filter-prune, the accuracy is {acc}%")

# Module 5: Iterative Pruning (20 pts)

In Module 4, we have implemented the filter-wise structral pruning strategy. The pruned model has a relative low sparsity level but a large degradation compared with magnitude pruning. A typical strategy to restore the accuracy is **Iterative Pruning**, within each iteration, two steps are involved:

*   Prune part of the weights
*   Retrain the remaining weight for accuracy recovery


Since we already know how to prune the model, in this Module, we will **dive into post-pruning retrain**.

## Question 5.1: Generate Mask (5 pts)

**Context & Motivation:**

When we retrain a pruned model, the primary challenge is ensuring that the pruned weights remain at zero. Even if a weight is successfully pruned, the standard optimization process will calculate a non-zero gradient for that position during backpropagation. Without intervention, the optimizer will use these gradients to update the weights, causing the "pruned" values to become non-zero again and effectively destroying the model's sparsity.

**Methods & Concepts:**

To maintain the pruned structure, we implement a **masking mechanism** on the gradients. By applying a binary mask (where 0 represents a pruned position and 1 represents a retained position) to the calculated gradients, we manually zero out the update signals for all pruned parameters.

**Tasks:**

You need to implement generate_mask function which generates masks for Conv2d. Return a dictionary where key is parameter name and value is mask tensor. The parameter name can be constructed via:

```
for name, module in model.named_modules():
  weight_param_name = f"{name}.weight"
  bias_param_name = f"{name}.bias"
```



For each output channel (or filter) i, compute its magnitude $s_i$, and define the mask as:
$$m_i = \mathbf{1}\left(s_i > \varepsilon\right)$$
where $\mathbf{1}(\cdot)$ is the indicator function.
If $s_i > \varepsilon$, set $m_i = 1$ (keep); otherwise $m_i = 0$ (prune).

For example, if the magnitudes for three filters are:
$s = [0.25, 0.003, 0.12]$ and $ϵ = 0.01$. Then the mask should be $m = [1, 0, 1]$.

In [ ]:
def generate_mask(model: nn.Module, eps: float = 1e-9):
    masks = {}
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
          ##############################################################################
          #TODO:
          # 1. Extract the weight tensor of the Conv2d layer.
          # 2. For each output channel (filter), determine whether the filter-wise sum is 0 or not
          # 3. Create mask:
          #    - mark as 1 if > eps (keep),
          #    - mark as 0 otherwise (prune).
          # 4. Reshape the 1D filter mask to 4D. The last three dimensions only contain 1 element.
          #    The result mask's shape should be (n, 1, 1, 1)
          # 5. If the layer has a bias, reuse the same 1D filter mask.
          # 6. Store the weight mask and its bias mask into masks dict. Using f"{name}.weight" and f"{name}.bias" as key.
          ##############################################################################



          ##############################################################################
          #END OF YOUR CODE
          ##############################################################################
        elif isinstance(module, nn.BatchNorm2d):
            bn_keep_mask = (torch.abs(module.weight.data) > eps).float()
            masks[f"{name}.weight"] = bn_keep_mask
            masks[f"{name}.bias"] = bn_keep_mask

    return masks

## Question 5.2: Apply Mask to Parameter (5 pts)

**Context & Motivation:**

After generating a mask, we must ensure it is applied to both the model's parameters and their corresponding gradients in each training step. Without a mask, the model can still generate non-zero gradient to the pruned weight which will make the weight non-pruned.

**Methods & Concepts:**

We use a unified function to apply the binary mask ($M$) to a single parameter ($W$) and its gradient ($\nabla$). This ensures that the pruned positions remain zeroed out in both the forward and backward passes. The logic follows two fundamental operations:


*   Weight Masking: $W_{\text{pruned}} = W \odot M$ (Ensures the model remains sparse)
*   Gradient Masking: $\nabla_{\text{pruned}} = \nabla \odot M$ (Ensures pruned weights are not updated)



**Tasks:**

You need to implement apply_mask function which applys mask to each parameter.

In [ ]:
def apply_mask(param: torch.Tensor, mask_tensor: torch.Tensor):
    with torch.no_grad():
        ##############################################################################
        #TODO:
        #1. Multiply the mask to the param (weight).
        #2. Multiply the mask to the gradident. You can access the gradient using param.grad.
        #   Be careful, the grad of the param may not exist such that param.grad is None.
        #   Make sure your implementation is aware of that, otherwise you will lose points.
        ##############################################################################



        ##############################################################################
        #END OF YOUR CODE
        ##############################################################################

## Question 5.3: Retrain Pipeline (10 pts)

**Context & Motivation:**

We have already implemented the necessary masking utility functions. The remaining task is to integrate them into the training loop. Without applying the mask during optimization, pruned weights will receive gradients and become non-zero again, breaking sparsity. The mask must be enforced at each update step.

**Methods & Concepts:**

To solve this, we must explicitly integrate our generate_mask and apply_mask functions into the training pipeline. First, generate_mask is used to create a dictionary mapping layer name to their mask tensor. Then, we manually call apply_mask in the training loop to ensure that both the weights ($W$) and the gradients ($\nabla$) are restricted by this mask.

**Tasks:**

You need to implement post_pruning_retrain function. In this function you have to complete three core logics:


1.   Generate masks given the pruned model.
2.   Gather parameters list used to initialize optimizer. To make this question simple, we only retrain the masked parameters.
3.   Apply apply_mask() function for parameters in finetune_params.



In [ ]:
def cross_entropy_loss(output: torch.Tensor, target: torch.Tensor):
  return F.cross_entropy(output, target)

def post_pruning_retrain(model: nn.Module,
                         dataloader: DataLoader,
                         loss_function: Callable,
                         device: str = 'cuda'):
    model.to(device)
    model.train()

    masks = None
    finetune_params = []
    ##############################################################################
    #TODO:
    #1. Call generate_mask to generate each layer's mask tensor. Assign the result to masks variable.
    #2. Iterate model's parameters.
    #   - If parameter's name is in masks variable's key set, set requires_grad = True, and add it to finetune_params.
    #   - If not, freeze the parameter (using requires_grad = False).
    ##############################################################################


    ##############################################################################
    #END OF YOUR CODE
    ##############################################################################
    optimizer = torch.optim.SGD(finetune_params, lr=1e-3, momentum=0.0, weight_decay=0.0)
    epochs = 10


    for _ in tqdm(range(epochs), leave=False):
        for batch in dataloader:
            x, y = batch
            x, y = x.to(device), y.to(device)

            out = model(x)
            loss = loss_function(out, y)

            optimizer.zero_grad()
            loss.backward()

            ##############################################################################
            #TODO:
            #1. Apply apply_mask() to parameters in finetune_params.
            ##############################################################################


            ##############################################################################
            #END OF YOUR CODE
            ##############################################################################

            optimizer.step()

    model.eval()
    return model

Let's retrain the pruned model and evalute it.

In [ ]:
retrained_model = copy.deepcopy(pruned_model)
retrained_model = post_pruning_retrain(retrained_model, dataloader['train'], cross_entropy_loss)

In [ ]:
acc = evaluate(retrained_model, dataloader['test'])
print(f"\nAfter retrain, the accuracy is {acc}%")

**Congratulations! You have successfully completed Lab 1!**